# Tool 3: Normalized Orderbook Viewer

## Purpose
Strips out price drift by normalizing all prices relative to the Wall Mid.
This transforms any product into a stationary view, revealing the true
microstructure patterns regardless of whether the price is moving or fixed.

## Why This Matters
Frankfurt Hedgehogs used this exact technique to discover that Kelp
(a moving-price product) "resembled Rainforest Resin [fixed price] but
with a tighter spread." Once normalized, you can see:
- The consistent bid-ask spread around fair value
- Where trades actually fill relative to the true price
- Whether a product is truly mean-reverting or just drifting

## What You'll See
- Y-axis = `price - wall_mid` (deviation from fair value)
- The zero line IS the fair price
- Blue dots below zero = bids (how far below fair the bids sit)
- Red dots above zero = asks (how far above fair the asks sit)
- Green triangles = trades (above zero = bought above fair, below = bought below)

## Key Insight
After normalization, a fixed-price product and a moving-price product
should look nearly identical — just a spread pattern around zero. If they
look different, that difference reveals something exploitable about the
moving product.

## Dependencies
- `data_loader.py`, `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PRODUCT = "ASH_COATED_OSMIUM"  # or "INTARIAN_PEPPER_ROOT"
DAY = -1

# Normalization baseline: "wall_mid" (recommended) or "raw_mid" or "mid_price"
NORMALIZE_BY = "wall_mid"

TIME_START = 0
TIME_END = None  # e.g., 100000

SHOW_TRADES = True
VOLUME_SCALE = 3
FIG_WIDTH = 20
FIG_HEIGHT = 7

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import (
    load_prices, load_trades, compute_wall_mid, compute_raw_mid,
    filter_time_range, PRODUCTS, AVAILABLE_DAYS,
)

prices = load_prices(day=DAY, product=PRODUCT)
prices = compute_wall_mid(prices)
prices = compute_raw_mid(prices)
trades = load_trades(day=DAY, product=PRODUCT)

if TIME_END is not None:
    prices = filter_time_range(prices, TIME_START, TIME_END)
    trades = filter_time_range(trades, TIME_START, TIME_END)

baseline = prices[NORMALIZE_BY].values
print(f"Normalizing {PRODUCT} (Day {DAY}) by {NORMALIZE_BY}")

In [ ]:
# ============================================================
# MAIN PLOT: Normalized Orderbook
# ============================================================
# Everything is plotted as deviation from the baseline (e.g., wall_mid).
# Zero = fair price. Positive = above fair. Negative = below fair.

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

ts = prices["timestamp"].values

# Bid levels (blue, below zero)
for level in [1, 2, 3]:
    p_col = f"bid_price_{level}"
    v_col = f"bid_volume_{level}"
    mask = prices[p_col].notna()
    if mask.any():
        norm_price = prices.loc[mask, p_col].values - baseline[mask]
        ax.scatter(
            ts[mask], norm_price,
            s=prices.loc[mask, v_col].values * VOLUME_SCALE,
            c="tab:blue", alpha=0.4, edgecolors="none",
            label=f"Bid L{level}" if level == 1 else None,
        )

# Ask levels (red, above zero)
for level in [1, 2, 3]:
    p_col = f"ask_price_{level}"
    v_col = f"ask_volume_{level}"
    mask = prices[p_col].notna()
    if mask.any():
        norm_price = prices.loc[mask, p_col].values - baseline[mask]
        ax.scatter(
            ts[mask], norm_price,
            s=prices.loc[mask, v_col].values * VOLUME_SCALE,
            c="tab:red", alpha=0.4, edgecolors="none",
            label=f"Ask L{level}" if level == 1 else None,
        )

# Trades (green triangles)
if SHOW_TRADES and len(trades) > 0:
    # Match each trade to nearest baseline value
    trade_baseline = np.interp(trades["timestamp"].values, ts, baseline)
    norm_trade_price = trades["price"].values - trade_baseline
    ax.scatter(
        trades["timestamp"], norm_trade_price,
        s=trades["quantity"] * VOLUME_SCALE * 3,
        c="tab:green", marker="^", alpha=0.7, edgecolors="black", linewidths=0.5,
        label="Trades", zorder=5,
    )

# Zero line = fair price
ax.axhline(0, color="orange", linewidth=1.5, linestyle="--", label=f"{NORMALIZE_BY} (=0)")

ax.set_xlabel("Timestamp", fontsize=12)
ax.set_ylabel(f"Price − {NORMALIZE_BY}", fontsize=12)
ax.set_title(f"Normalized Orderbook — {PRODUCT} (Day {DAY})", fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPARISON: Both Products Normalized Side-by-Side
# ============================================================
# If both products look similar after normalization, they can be
# traded with the same market-making logic. Differences reveal
# unique characteristics to exploit.

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

for idx, prod in enumerate(PRODUCTS):
    ax = axes[idx]
    p = compute_wall_mid(compute_raw_mid(load_prices(day=DAY, product=prod)))
    t = load_trades(day=DAY, product=prod)
    bl = p[NORMALIZE_BY].values
    ts_p = p["timestamp"].values

    for level in [1, 2, 3]:
        mask = p[f"bid_price_{level}"].notna()
        if mask.any():
            ax.scatter(ts_p[mask], p.loc[mask, f"bid_price_{level}"].values - bl[mask],
                       s=p.loc[mask, f"bid_volume_{level}"].values * VOLUME_SCALE,
                       c="tab:blue", alpha=0.3, edgecolors="none")
        mask = p[f"ask_price_{level}"].notna()
        if mask.any():
            ax.scatter(ts_p[mask], p.loc[mask, f"ask_price_{level}"].values - bl[mask],
                       s=p.loc[mask, f"ask_volume_{level}"].values * VOLUME_SCALE,
                       c="tab:red", alpha=0.3, edgecolors="none")

    if len(t) > 0:
        t_bl = np.interp(t["timestamp"].values, ts_p, bl)
        ax.scatter(t["timestamp"], t["price"].values - t_bl,
                   s=t["quantity"] * VOLUME_SCALE * 3,
                   c="tab:green", marker="^", alpha=0.6, edgecolors="black", linewidths=0.3)

    ax.axhline(0, color="orange", linewidth=1.5, linestyle="--")
    ax.set_title(f"{prod} (Day {DAY})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel(f"Price − {NORMALIZE_BY}")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()